# ChargeSquare Analytics — A1–A6

`energy_kwh` is the **cumulative** per-session meter register — the classic **energy trap**.
Every energy metric below uses per-session deltas, **never** `SUM(energy_kwh)`, which would
over-count by ~(readings per session).

This notebook runs the six analytical queries in `analytics/queries/` against ClickHouse
(over its HTTP interface, port 8123) and writes each result to `analytics/output/*.csv`.


In [1]:
import os
from pathlib import Path
import clickhouse_connect
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Resolve the pipeline root robustly: honour $PIPELINE_ROOT if set, otherwise walk
# upward from the current directory until we find the one that holds BOTH
# docker-compose.yml and analytics/queries. Works from the repo root, from analytics/,
# or from any nested subdirectory.
def find_root():
    env = os.environ.get("PIPELINE_ROOT")
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "docker-compose.yml").exists() and (d / "analytics/queries").is_dir():
            return d
    raise RuntimeError(
        "could not locate pipeline root (need docker-compose.yml + analytics/queries); "
        "set PIPELINE_ROOT to override"
    )

ROOT = find_root()
QUERIES = ROOT / "analytics/queries"
OUTPUT = ROOT / "analytics/output"
OUTPUT.mkdir(parents=True, exist_ok=True)

client = clickhouse_connect.get_client(
    host=os.environ.get("CLICKHOUSE_HOST", "localhost"),
    port=int(os.environ.get("CLICKHOUSE_PORT", "8123")),
    username="chargesquare", password="chargesquare", database="ev",
)

LABELS = ["A1", "A2", "A3", "A4", "A5", "A6"]

def run(label):
    # clickhouse-connect wants a single statement, so strip the trailing ';'.
    sql = sorted(QUERIES.glob(f"{label}_*.sql"))[0].read_text(encoding="utf-8").strip().rstrip(";")
    df = client.query_df(sql)
    df.to_csv(OUTPUT / f"{label}.csv", index=False)
    print(f"{label}: {len(df)} rows -> analytics/output/{label}.csv")
    return df

def save_plot(name):
    try:
        plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()
    except Exception as e:
        print("plot skipped:", e)
    finally:
        plt.close("all")


## A1 — Hourly energy consumption (per-session deltas, not cumulative sums)


In [2]:
a1 = run("A1")
try:
    a1.plot(x="hour", y="energy_kwh", figsize=(10,3), legend=False, title="A1 Hourly energy (kWh)")
    save_plot("A1")
except Exception as e:
    print("plot skipped:", e)
a1.head()


A1: 16 rows -> analytics/output/A1.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_41509/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,hour,energy_kwh
0,2026-07-02 18:00:00,16819.151
1,2026-07-02 19:00:00,31779.534
2,2026-07-02 20:00:00,26859.547
3,2026-07-02 21:00:00,26407.145
4,2026-07-02 22:00:00,27299.498


## A2 — Station uptime / downtime, worst stations per operator


In [3]:
a2 = run("A2")
try:
    top = a2.head(12).iloc[::-1]
    top.plot(kind="barh", x="station_id", y="downtime_s", figsize=(10,4), legend=False, title="A2 Downtime seconds (top stations)")
    save_plot("A2")
except Exception as e:
    print("plot skipped:", e)
a2.head(12)


A2: 20 rows -> analytics/output/A2.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_41509/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,operator_id,station_id,total_s,downtime_s,uptime_ratio
0,VoltRun,TR-IZM-0002,220652,8601,0.9610
1,Trugo,TR-IZM-0020,220652,6399,0.9710
2,Esarj,TR-IZM-0017,220652,6342,0.9713
3,ZES,TR-ANK-0012,220652,6226,0.9718
4,ChargeSquare,TR-IST-0004,165489,6152,0.9628
5,VoltRun,TR-IST-0062,220652,6141,0.9722
6,Esarj,TR-ANT-0012,220652,6110,0.9723
7,VoltRun,TR-IZM-0019,220652,6000,0.9728
8,Trugo,TR-IZM-0010,110326,5922,0.9463
9,ZES,TR-IZM-0005,165489,5520,0.9666


## A3 — Average duration & energy by vehicle brand


In [4]:
a3 = run("A3")
try:
    a3.plot(kind="bar", x="brand", y=["avg_duration_min","avg_energy_kwh"], figsize=(10,3), title="A3 Avg duration (min) & energy (kWh) by brand")
    save_plot("A3")
except Exception as e:
    print("plot skipped:", e)
a3


A3: 9 rows -> analytics/output/A3.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_41509/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,brand,sessions,avg_duration_min,avg_energy_kwh
0,Tesla,5130,27.5,33.56
1,Volvo,1777,30.6,34.08
2,Volkswagen,1022,33.0,36.22
3,Hyundai,985,29.2,37.07
4,Togg,741,31.8,38.26
5,Kia,725,29.6,36.12
6,BMW,647,30.8,36.66
7,Renault,592,30.8,29.15
8,Mercedes-Benz,499,34.0,32.23


## A4 — Revenue and peak-rate contribution, per operator


In [5]:
a4 = run("A4")
try:
    a4.plot(kind="bar", x="operator_id", y=["revenue_eur","peak_revenue_eur"], figsize=(10,3), title="A4 Revenue (EUR): total vs peak-rate")
    save_plot("A4")
except Exception as e:
    print("plot skipped:", e)
a4


A4: 132 rows -> analytics/output/A4.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_41509/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,operator_id,city,tariff_id,revenue_eur,peak_revenue_eur,peak_pct,sessions
0,ChargeSquare,Istanbul,standard-v1,7505.33,695.34,9.3,529
1,VoltRun,Istanbul,standard-v1,6713.60,547.50,8.2,488
2,ZES,Istanbul,standard-v1,5991.65,462.07,7.7,454
3,Esarj,Istanbul,standard-v1,5298.75,293.56,5.5,400
4,VoltRun,Izmir,standard-v1,5032.06,309.17,6.1,370
...,...,...,...,...,...,...,...
127,ChargeSquare,Bursa,off-peak-v1,121.86,0.00,0.0,12
128,VoltRun,Konya,off-peak-v1,105.20,8.65,8.2,23
129,Trugo,Antalya,fleet-v1,75.39,0.00,0.0,5
130,ZES,Adana,peak-rate-v2,75.05,55.55,74.0,3


## A5 — Geographic distribution of FAULT events (deduped by event_id)


In [6]:
a5 = run("A5")
try:
    a5.plot(kind="bar", x="city", y="fault_count", figsize=(10,3), legend=False, title="A5 Fault count by city")
    save_plot("A5")
except Exception as e:
    print("plot skipped:", e)
a5


A5: 7 rows -> analytics/output/A5.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_41509/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,city,fault_count,stations_affected,faults_per_station,lat,lon
0,Istanbul,143,66,2.17,41.0049,28.9805
1,Izmir,77,27,2.85,38.4211,27.1326
2,Ankara,75,30,2.50,39.9070,32.8770
3,Antalya,35,15,2.33,36.8786,30.7049
4,Adana,28,10,2.80,37.0171,35.3370
5,Bursa,14,8,1.75,40.1788,29.0651
6,Konya,12,5,2.40,37.8579,32.5129


## A6 — Anomaly detection: sessions > 2 sigma above the fleet mean power


In [7]:
a6 = run("A6")
try:
    a6.head(20).plot(kind="bar", x="session_id", y="z_score", figsize=(10,3), legend=False, title="A6 Anomalous sessions (z-score)")
    save_plot("A6")
except Exception as e:
    print("plot skipped:", e)
a6.head(20)


A6: 100 rows -> analytics/output/A6.csv


/var/folders/88/5j93ks7j4v3d_pf8hqr0_prr0000gn/T/ipykernel_41509/741225504.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUTPUT / f"{name}.png"); plt.show()


,session_id,station_id,brand,avg_power_kw,z_score
0,sess-e3663911-a840-4bd9,TR-ANK-0032,Tesla,250.0,3.05
1,sess-d815b2de-4b9b-4241,TR-ANK-0007,Tesla,250.0,3.05
2,sess-168d5ba1-a529-4def,TR-IST-0042,Tesla,250.0,3.05
3,sess-a6ef5ce9-1bf1-4568,TR-IST-0069,Tesla,250.0,3.05
4,sess-4c706c80-0915-40c6,TR-ANK-0004,Tesla,250.0,3.05
5,sess-b9d5b95b-d536-4488,TR-IST-0007,Tesla,250.0,3.05
6,sess-da2dc887-6377-4d65,TR-BUR-0010,Tesla,250.0,3.05
7,sess-18988476-a517-4933,TR-IZM-0028,Tesla,250.0,3.05
8,sess-59f05996-9a6c-4997,TR-IST-0083,Tesla,250.0,3.05
9,sess-3f398a60-65de-4fca,TR-IST-0025,Tesla,250.0,3.05


## Summary

All six results are in `analytics/output/*.csv`. The energy figures (A1, A3) use
per-session **deltas** of the cumulative `energy_kwh` register — summing the raw column
would over-count by roughly the number of meter readings per session.
